# Data Storm 7.0 — InsightAI

> **Competition:** Data Storm 7.0 — Storming Round (36-hour hackathon)

> **Team:** InsightAI

> **Objective:** Estimate the latent maximum monthly volume potential (in liters) for ~20,000 retail outlets for January 2026.

In [7]:
#importing necessary libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
#setup paths

BASE_PATH = Path("../src/bronze")

TRANSACTION_PATH = BASE_PATH / "transactions_history_final.csv"
OUTLET_MASTER_PATH = BASE_PATH / "outlet_master.csv"
COORDINATE_PATH = BASE_PATH / "outlet_coordinates.csv"
HOLIDAY_PATH = BASE_PATH / "holiday_list.csv"
SEASONALITY_PATH = BASE_PATH / "distributor_seasonality_details.csv"

In [9]:
#Loading datasets

transactions_df = pd.read_csv(TRANSACTION_PATH)
outlet_master_df = pd.read_csv(OUTLET_MASTER_PATH)
coordinates_df = pd.read_csv(COORDINATE_PATH)
holiday_df = pd.read_csv(HOLIDAY_PATH)
seasonality_df = pd.read_csv(SEASONALITY_PATH)

In [10]:
# Dataset Collection

datasets = {
    "transactions": transactions_df,
    "outlet_master": outlet_master_df,
    "coordinates": coordinates_df,
    "holidays": holiday_df,
    "seasonality": seasonality_df
}

In [11]:
# Basic overview

for name, df in datasets.items():

    print("=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)

    print(f"Shape: {df.shape}")

    display(df.head())

    print("\n")

DATASET: transactions
Shape: (2376389, 7)


,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value
0,OUT_19886,2024,12,DIST_S_02,SKU_10,5.897879,2177.632359
1,OUT_00837,2024,2,DIST_W_01,SKU_03,20.697364,7244.084814
2,OUT_15438,2025,12,DIST_NW_01,SKU_02,55.101801,13959.108787
3,OUT_12992,2025,1,DIST_C_01,SKU_07,24.063953,15641.548773
4,OUT_12334,2025,5,DIST_C_02,SKU_04,47.769665,15525.158656




DATASET: outlet_master
Shape: (20000, 4)


,Outlet_ID,Outlet_Size,Cooler_Count,Outlet_Type
0,OUT_00001,Medium,1,Grocry
1,OUT_00002,Small,0,Hotel
2,OUT_00003,Small,1,Pharmacy
3,OUT_00004,Medium,2,Pharmacy
4,OUT_00005,Medium,2,Kiosk




DATASET: coordinates
Shape: (20000, 3)


,Outlet_ID,Latitude,Longitude
0,OUT_00001,7.089846,79.979055
1,OUT_00002,7.000558,80.012422
2,OUT_00003,6.806170,79.854547
3,OUT_00004,6.703533,79.806919
4,OUT_00005,7.186878,79.869831




DATASET: holidays
Shape: (349, 3)


,Date,Holiday_Name,Holiday_Type
0,2023-01-06T00:00:00Z,Duruthu Full Moon Poya Day,Public
1,2023-01-15T00:00:00Z,Tamil Thai Pongal Day,Public
2,2023-01-16T00:00:00Z,Additional Holiday in lieu of Tamil Thai Ponga...,Public
3,2023-02-03T00:00:00Z,Additional Half Holiday in lieu of the Indepen...,Public
4,2023-02-04T00:00:00Z,National Day,Public




DATASET: seasonality
Shape: (360, 4)


,Distributor_ID,Year,Month,Seasonality_Index
0,DIST_W_01,2023,1,Moderate
1,DIST_W_01,2023,2,Moderate
2,DIST_W_01,2023,3,Moderate
3,DIST_W_01,2023,4,Favorable
4,DIST_W_01,2023,5,Un-Favorable


In [ ]:
# Clumn and data type inspection

for name, df in datasets.items():

    print("=" * 60)
    print(f"INFO: {name}")
    print("=" * 60)

    print(df.info())

    print("\n")

INFO: transactions
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2376389 entries, 0 to 2376388
Data columns (total 7 columns):
 #   Column            Dtype  
---  ------            -----  
 0   Outlet_ID         object 
 1   Year              int64  
 2   Month             int64  
 3   Distributor_ID    object 
 4   SKU_ID            object 
 5   Volume_Liters     float64
 6   Total_Bill_Value  float64
dtypes: float64(2), int64(2), object(3)
memory usage: 126.9+ MB
None


INFO: outlet_master
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Outlet_ID     20000 non-null  object
 1   Outlet_Size   19804 non-null  object
 2   Cooler_Count  20000 non-null  int64 
 3   Outlet_Type   20000 non-null  object
dtypes: int64(1), object(3)
memory usage: 625.1+ KB
None


INFO: coordinates
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entrie

In [14]:
# Missing Value analysis

for name, df in datasets.items():

    print("=" * 60)
    print(f"MISSING VALUES: {name}")
    print("=" * 60)

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values.\n")

    else:
        display(
            missing.sort_values(ascending=False)
        )

    print("\n")


MISSING VALUES: transactions
No missing values.



MISSING VALUES: outlet_master


Outlet_Size    196
dtype: int64



MISSING VALUES: coordinates
No missing values.



MISSING VALUES: holidays
No missing values.



MISSING VALUES: seasonality
No missing values.





In [15]:
# duplicate analysis

for name, df in datasets.items():

    print("=" * 60)
    print(f"DUPLICATES: {name}")
    print("=" * 60)

    duplicate_count = df.duplicated().sum()

    print(f"Duplicate Rows: {duplicate_count}")

    print("\n")

DUPLICATES: transactions
Duplicate Rows: 0


DUPLICATES: outlet_master
Duplicate Rows: 0


DUPLICATES: coordinates
Duplicate Rows: 0


DUPLICATES: holidays
Duplicate Rows: 93


DUPLICATES: seasonality
Duplicate Rows: 0




In [31]:
# see duplicating rows

duplicate_rows = holiday_df[holiday_df.duplicated(keep=False)]
print(duplicate_rows.sort_values(by=['Date' , 'Holiday_Name', 'Holiday_Type']))

                     Date                Holiday_Name Holiday_Type
47   2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day         Bank
223  2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day         Bank
73   2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day   Mercantile
248  2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day   Mercantile
26   2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day     Poya Day
..                    ...                         ...          ...
136  2024-12-14T00:00:00Z  Unduvap Full Moon Poya Day     Poya Day
27   2025-02-12T00:00:00Z    Navam Full Moon Poya Day     Poya Day
295  2025-02-12T00:00:00Z    Navam Full Moon Poya Day     Poya Day
36   2025-08-10T00:00:00Z   Nikini Full Moon Poya Day     Poya Day
303  2025-08-10T00:00:00Z   Nikini Full Moon Poya Day     Poya Day

[186 rows x 3 columns]


In [24]:
print(holiday_df.loc[[0, 185]])

                     Date                Holiday_Name Holiday_Type
0    2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day       Public
185  2023-01-06T00:00:00Z  Duruthu Full Moon Poya Day       Public


**this mean there are some exact duplicates but some are not exact duplicates.**

In [26]:
# Numerical summary

for name, df in datasets.items():

    print("=" * 60)
    print(f"NUMERICAL SUMMARY: {name}")
    print("=" * 60)

    # Check if there are any actual numeric columns in this dataframe
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if not numeric_cols.empty:
        display(df[numeric_cols].describe())
    else:
        print(f"'{name}' does not contain any numerical columns.")

    print("\n")

NUMERICAL SUMMARY: transactions


,Year,Month,Volume_Liters,Total_Bill_Value
count,2.376389e+06,2.376389e+06,2.376389e+06,2.376389e+06
mean,2.024000e+03,6.499014e+00,5.262422e+01,1.379062e+04
std,8.165350e-01,3.449912e+00,9.548678e+01,1.644881e+04
min,2.023000e+03,1.000000e+00,-9.564408e+02,-1.411536e+05
25%,2.023000e+03,3.000000e+00,1.018487e+01,3.527437e+03
50%,2.024000e+03,6.000000e+00,2.315845e+01,8.060190e+03
75%,2.025000e+03,9.000000e+00,5.443157e+01,1.629443e+04
max,2.025000e+03,1.200000e+01,9.438578e+03,1.528457e+05




NUMERICAL SUMMARY: outlet_master


,Cooler_Count
count,20000.000000
mean,1.291000
std,1.375505
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,5.000000




NUMERICAL SUMMARY: coordinates


,Latitude,Longitude
count,20000.000000,20000.000000
mean,7.739852,79.285245
std,7.303732,8.103265
min,0.000000,0.000000
25%,6.811049,79.925880
50%,7.022154,80.058188
75%,7.350576,80.456441
max,80.792317,80.799952




NUMERICAL SUMMARY: holidays
'holidays' does not contain any numerical columns.


NUMERICAL SUMMARY: seasonality


,Year,Month
count,360.000000,360.000000
mean,2024.000000,6.500000
std,0.817633,3.456857
min,2023.000000,1.000000
25%,2023.000000,3.750000
50%,2024.000000,6.500000
75%,2025.000000,9.250000
max,2025.000000,12.000000


In [27]:
# categorical summary 

for name, df in datasets.items():

    print("=" * 60)
    print(f"CATEGORICAL SUMMARY: {name}")
    print("=" * 60)

    categorical_cols = df.select_dtypes(
        include=['object']
    ).columns

    for col in categorical_cols:

        print(f"\nCOLUMN: {col}")

        print(df[col].value_counts().head(10))

    print("\n")

CATEGORICAL SUMMARY: transactions

COLUMN: Outlet_ID
Outlet_ID
OUT_11497    367
OUT_06356    367
OUT_06981    366
OUT_12492    365
OUT_13808    365
OUT_11251    365
OUT_07161    365
OUT_00815    364
OUT_11843    364
OUT_11094    364
Name: count, dtype: int64

COLUMN: Distributor_ID
Distributor_ID
DIST_W_01     366300
DIST_W_02     365220
DIST_W_03     363012
DIST_NW_01    239818
DIST_NW_02    236101
DIST_S_02     168795
DIST_S_01     167376
DIST_C_01     164284
DIST_C_03     153129
DIST_C_02     152354
Name: count, dtype: int64

COLUMN: SKU_ID
SKU_ID
SKU_01    287644
SKU_07    232648
SKU_10    232336
SKU_04    232247
SKU_06    232066
SKU_03    232005
SKU_02    231963
SKU_05    231867
SKU_08    231828
SKU_09    231785
Name: count, dtype: int64


CATEGORICAL SUMMARY: outlet_master

COLUMN: Outlet_ID
Outlet_ID
OUT_00001    1
OUT_00002    1
OUT_00003    1
OUT_00004    1
OUT_00005    1
OUT_00006    1
OUT_00007    1
OUT_00008    1
OUT_00009    1
OUT_00010    1
Name: count, dtype: int64

COLU

In [30]:
# export audit reports

audit_output_path = Path("../output")

audit_output_path.mkdir(
    exist_ok=True
)

summary = []

for name, df in datasets.items():

    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicates": df.duplicated().sum(),
        "missing_values": df.isnull().sum().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    audit_output_path / "dataset_audit_summary.csv",
    index=False
)

print("Audit summary exported.")

Audit summary exported.
